# Enriched rectangular RT elements

Normal degree and interior degree are independent. The stored six-level comparison uses the smooth NeoPZ labmec/MHM data and the published Figure 3 markers, with raster uncertainty retained.

The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/e2f36b29d4ac8821ffc752b56b5dc52adc024f2e5532413e210b823407dd75d6/25_tensor_rt-companion.zip"
COMPANION_SHA256 = "e2f36b29d4ac8821ffc752b56b5dc52adc024f2e5532413e210b823407dd75d6"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/25_tensor_rt.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


## Declare the physical variational forms

The application defines local operators with the shared element, integration
and oriented face kernels and supplies the actual A/B/C blocks to the generic
assembler. For primal pressure the form is (K grad(p),grad(v)); mixed coordinates
use the inverse-permeability flux mass, divergence and normal-moment constraint.

$$
\begin{aligned}
A_Tu_T+B_T\lambda&=f_T,\\
C_Tu_T+D_T\lambda&=g_T, & C_T&=-B_T^{\mathsf T}, & D_T&=0.
\end{aligned}
$$

The local constant pressure and its physical integral moment retain the executed
nodal/modal basis. The separate global `Equation` supplies the signed pressure
boundary functional; pure Neumann data add a physical pressure integral constraint.
The flux trace follows fixed macroface normals. Mixed flux coordinates retain
their H(div) normal/interior moments and Piola transform; a primal gradient
field is evaluated independently. Generic assembly/solve and physical coefficient
interpretation are separate operations.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.tensor_darcy import define_tensor_darcy, recover_tensor_darcy

from pymhm import CartesianMacroMesh

mesh = CartesianMacroMesh(2)
solution_definition = define_tensor_darcy(mesh, degree=1, enrichment=1, dirichlet=lambda x: x[:, 0])
solution_system = assemble(solution_definition.problem)
solution = recover_tensor_darcy(
    solution_definition,
    solution_system,
    solution_system.solve(constraints=pressure_constraints(solution_definition, solution_system)),
)
errors = solution.errors(lambda x: x[:, 0], (-1.0, 0.0), 0.0)
assert max(errors.values()) < 1e-11
errors


The full research campaign is acquired separately from this lightweight notebook. The following cells display its numerical record and publication figures.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    from pathlib import Path
    import json
    from IPython.display import Image, display
    record = json.loads((root / "examples/results/tensor-rt-comparison.json").read_text())
    record


In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(root / "docs/figures/tensor-rt/convergence.png")))
